# 11 – Concatenation Fusion (ablation control)

Ablation control for the fusion mechanism: identical backbone, features and training
protocol as the cross-attention model (notebook 10), but the two modality representations
are combined by simple concatenation instead of attention. Includes a five-seed sensitivity
analysis.

**Run after notebooks 05, 06 and 09b** — uses hourly_vitals.csv,
modelling_cohort_sepsis_mortality.csv, text_hourly_cls_mbert.npz.

**Produces:** preds_concat.npz (used by notebook 13).

MIMIC-III data not included (PhysioNet DUA); see README.

In [ ]:
# --- Setup ---
import os

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

import numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Load data (identical to nb 10)

In [ ]:
tz = np.load(data_path("text_hourly_cls_mbert.npz"), allow_pickle=True)
X_text_all, has_note_all, text_ids = tz["X_text"], tz["has_note"], tz["stay_ids"]

vitals = pd.read_csv(data_path("hourly_vitals.csv"))
cohort = pd.read_csv(data_path("modelling_cohort_sepsis_mortality.csv"))
labels = cohort[["ICUSTAY_ID","mortality_after_24h"]].drop_duplicates()

VITAL_COLS = [
    "heart_rate",
    "sbp",
    "dbp",
    "map",
    "resp_rate",
    "temperature",
    "spo2",
    "fio2",
    "glucose",
    "ph",
    "gcs_eye",
    "gcs_motor",
    "gcs_total",
    "gcs_verbal",
    "weight",
    "height"
]

MASK_COLS = [c + "_observed" for c in VITAL_COLS]
FEAT_COLS = VITAL_COLS + MASK_COLS
stay_ids=text_ids; y=labels.set_index("ICUSTAY_ID")["mortality_after_24h"]

vitals=vitals[vitals["ICUSTAY_ID"].isin(set(stay_ids))].copy()
full_idx=pd.MultiIndex.from_product([stay_ids,range(24)],names=["ICUSTAY_ID","ICU_HOUR"])
vit=vitals.set_index(["ICUSTAY_ID","ICU_HOUR"]).reindex(full_idx)[FEAT_COLS]
X_sig_raw=vit[FEAT_COLS].values.reshape(len(stay_ids),24,len(FEAT_COLS)).astype("float32")

## 2. Same split + standardisation as nb 10

In [ ]:
train_ids,temp_ids=train_test_split(stay_ids,test_size=0.30,random_state=42,stratify=y.loc[stay_ids])
val_ids,test_ids=train_test_split(temp_ids,test_size=0.50,random_state=42,stratify=y.loc[temp_ids])
pos={s:i for i,s in enumerate(stay_ids)}; idx=lambda ids:[pos[s] for s in ids]

n_vitals = len(VITAL_COLS)

tr_idx = idx(train_ids)

tr_vals = (
    X_sig_raw[tr_idx][:, :, :n_vitals]
    .reshape(-1, n_vitals)
)

mu = np.nanmean(tr_vals, axis=0)
sd = np.nanstd(tr_vals, axis=0)
sd[sd == 0] = 1

X_sig = X_sig_raw.copy()

X_sig[:, :, :n_vitals] = (
    X_sig[:, :, :n_vitals] - mu
) / sd

X_sig = np.nan_to_num(X_sig, nan=0.0)

print("signal shape:", X_sig.shape)

def make(ids):
    i=idx(ids)
    return (torch.tensor(X_sig[i]),torch.tensor(X_text_all[i]),
            torch.tensor(has_note_all[i]),torch.tensor(y.loc[ids].values.astype("float32")))
Xs_tr,Xt_tr,m_tr,y_tr=make(train_ids); Xs_va,Xt_va,m_va,y_va=make(val_ids); Xs_te,Xt_te,m_te,y_te=make(test_ids)
tr_loader=DataLoader(TensorDataset(Xs_tr,Xt_tr,m_tr,y_tr),batch_size=128,shuffle=True)

signal shape: (10068, 24, 32)


## 3. Concatenation fusion model
Same signal LSTM and text projection as nb 10, but instead of cross-attention the two
modality representations are **concatenated** and passed to the classifier.

In [ ]:
class ConcatFusion(nn.Module):
    def __init__(
        self,
        sig_dim=len(FEAT_COLS),
        text_dim=768,
        d=128,
        dropout=0.3
    ):
        super().__init__()
        self.sig_lstm = nn.LSTM(sig_dim, d, batch_first=True)   # same as nb10
        self.text_proj = nn.Linear(text_dim, d)                 # same as nb10
        self.fc = nn.Sequential(nn.Linear(d*2, 64), nn.ReLU(),  # d*2 because concatenated
                                nn.Dropout(dropout), nn.Linear(64,1))
    def forward(self, sig, text, note_mask):
        s,_ = self.sig_lstm(sig)                 # (B,24,d)
        s_rep = s.mean(dim=1)                    # pool signal -> (B,d)
        t = self.text_proj(text)                 # (B,24,d)
        # masked mean pool over hours with a note (ignore empty hours)
        m = note_mask.unsqueeze(-1)
        t_rep = (t*m).sum(1) / m.sum(1).clamp(min=1)   # (B,d)
        fused = torch.cat([s_rep, t_rep], dim=1)       # concatenation (no attention)
        return self.fc(fused).squeeze(1)

import random
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model = ConcatFusion().to(device)

## 4. Train (identical protocol to nb 10)

In [ ]:
pr=float(y_tr.mean()); pw=torch.tensor([(1-pr)/pr],device=device)
crit=nn.BCEWithLogitsLoss(pos_weight=pw); opt=torch.optim.Adam(model.parameters(),lr=1e-3)

@torch.no_grad()
def ev(Xs,Xt,m,yv):
    model.eval()
    p=torch.sigmoid(model(Xs.to(device),Xt.to(device),m.to(device))).cpu().numpy(); yt=yv.numpy()
    return roc_auc_score(yt,p),average_precision_score(yt,p),f1_score(yt,(p>=0.5).astype(int))

best,bs=0,None
for ep in range(1,61):
    model.train()
    for xs,xt,mm,yb in tr_loader:
        xs,xt,mm,yb=xs.to(device),xt.to(device),mm.to(device),yb.to(device)
        opt.zero_grad(); crit(model(xs,xt,mm),yb).backward(); opt.step()
    va,_,_=ev(Xs_va,Xt_va,m_va,y_va)
    if ep%10==0: print(f"epoch {ep} | val AUROC {va:.3f}")
    if va>best: best,bs=va,{k:v.cpu().clone() for k,v in model.state_dict().items()}
model.load_state_dict(bs)

epoch 10 | val AUROC 0.793
epoch 20 | val AUROC 0.776
epoch 30 | val AUROC 0.739
epoch 40 | val AUROC 0.736
epoch 50 | val AUROC 0.738
epoch 60 | val AUROC 0.732


<All keys matched successfully>

## 5. Full ablation table

In [ ]:
au,ap,f1=ev(Xs_te,Xt_te,m_te,y_te)
print("=== Concatenation fusion (test) ===")
print(f"AUROC: {au:.3f}  AUPRC: {ap:.3f}  F1: {f1:.3f}")
print("\n=== FULL ABLATION TABLE ===")
print(f"{'model':<26}{'AUROC':<8}{'AUPRC':<8}{'F1':<8}")
print(f"{'signal-only':<26}{'0.675':<8}{'0.379':<8}{'0.413':<8}")
print(f"{'text-only':<26}{'0.669':<8}{'0.337':<8}{'0.397':<8}")
print(f"{'concat fusion':<26}{au:<8.3f}{ap:<8.3f}{f1:<8.3f}")
print(f"{'cross-attention fusion':<26}{'0.794':<8}{'0.512':<8}{'0.515':<8}")

=== Concatenation fusion (test) ===
AUROC: 0.786  AUPRC: 0.507  F1: 0.487

=== FULL ABLATION TABLE ===
model                     AUROC   AUPRC   F1      
signal-only               0.675   0.379   0.413   
text-only                 0.669   0.337   0.397   
concat fusion             0.786   0.507   0.487   
cross-attention fusion    0.794   0.512   0.515   


In [ ]:
@torch.no_grad()
def predict_probs(Xs, Xt, m):
    model.eval()
    return torch.sigmoid(model(Xs.to(device), Xt.to(device), m.to(device))).cpu().numpy()

p_te = predict_probs(Xs_te, Xt_te, m_te)
np.savez(data_path("preds_concat.npz"),
         stay_ids=np.array(test_ids), y_true=y_te.numpy(), y_prob=p_te)
print("saved preds_concat.npz", p_te.shape)

saved preds_concat.npz (1511,)


In [ ]:
# ===== Multi-seed sensitivity analysis (Appendix D) =====
import numpy as np, torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

SEEDS = [42, 1, 2, 3, 4]
_res = []

def _ev_test(m):
    m.eval()
    with torch.no_grad():
        p = torch.sigmoid(m(Xs_te.to(device), Xt_te.to(device), m_te.to(device))).cpu().numpy()
    yt = y_te.numpy()
    return (roc_auc_score(yt, p), average_precision_score(yt, p),
            f1_score(yt, (p >= 0.5).astype(int)))

def _ev_val(m):
    m.eval()
    with torch.no_grad():
        p = torch.sigmoid(m(Xs_va.to(device), Xt_va.to(device), m_va.to(device))).cpu().numpy()
    return roc_auc_score(y_va.numpy(), p)

for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    m = ConcatFusion().to(device)
    pr = float(y_tr.mean()); pw = torch.tensor([(1-pr)/pr], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    best, bs = 0, None
    for ep in range(1, 61):
        m.train()
        for xs, xt, mm, yb in tr_loader:
            xs, xt, mm, yb = xs.to(device), xt.to(device), mm.to(device), yb.to(device)
            opt.zero_grad(); crit(m(xs, xt, mm), yb).backward(); opt.step()
        va = _ev_val(m)
        if va > best: best, bs = va, {k:v.cpu().clone() for k,v in m.state_dict().items()}
    m.load_state_dict(bs)
    au, ap, f1 = _ev_test(m)
    _res.append((au, ap, f1)); print(f"seed {seed}: AUROC {au:.3f} AUPRC {ap:.3f} F1 {f1:.3f}")

_res = np.array(_res)
print(f"\nConcat  AUROC {_res[:,0].mean():.3f} ± {_res[:,0].std():.3f}"
      f" | AUPRC {_res[:,1].mean():.3f} ± {_res[:,1].std():.3f}"
      f" | F1 {_res[:,2].mean():.3f} ± {_res[:,2].std():.3f}")

seed 42: AUROC 0.786 AUPRC 0.507 F1 0.487
seed 1: AUROC 0.786 AUPRC 0.503 F1 0.510
seed 2: AUROC 0.785 AUPRC 0.497 F1 0.508
seed 3: AUROC 0.788 AUPRC 0.509 F1 0.533
seed 4: AUROC 0.790 AUPRC 0.510 F1 0.514

Concat  AUROC 0.787 ± 0.002 | AUPRC 0.505 ± 0.005 | F1 0.510 ± 0.014
